In [ ]:
#!/usr/bin/env python3
"""
DREAM Cloud Analysis Pipeline  —  with OpSim-consistent Seeing + M5 Correction
================================================================================

What changed vs the original pipeline
--------------------------------------
Every occurrence of the old photon-proxy depth estimate has been replaced with
a physically correct, OpSim-self-consistent 5-sigma limiting magnitude:

  Old (proxy):
      photons = flux * area * QE * expose_t
      depth   = photons_to_magnitude(photons)   ← made-up SNR formula

  New (correct):
      1. For every observation slot, find the temporally-nearest OpSim row
         (same night, same band) → grab seeingFwhm500, skyBrightness, airmass.
      2. Run rubin_scheduler.site_models.SeeingModel(fwhm_z, airmass)
         → fwhmEff  (telescope + optics + camera + atmosphere, wavelength-scaled)
      3. Run rubin_scheduler.utils.m5_flat_sed(band, musky, fwhmEff,
                                               exptime, airmass,
                                               tau_cloud=dream_extinction)
         → m5_corrected  (native tau_cloud path — correct noise propagation)

Keys added to the metrics dict per strategy
-------------------------------------------
  "fwhmEff"       list[float]   arcsec, one per slot
  "m5_clear"      list[float]   mag,    clear-sky depth (tau_cloud=0)
  "m5_corrected"  list[float]   mag,    cloud-corrected depth
  "sky_bright"    list[float]   mag/arcsec², musky from OpSim
  "seeing500"     list[float]   arcsec, seeingFwhm500 from OpSim

The old "photons" list is kept for backward compat but is no longer used
for depth estimation.  Slew-gating logic is unchanged.
"""

import io
import os
import re
import sqlite3
import warnings
import inspect
import numpy as np
import pandas as pd
import h5py
import healpy as hp
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import matplotlib.dates as mdates
from matplotlib.patches import FancyArrowPatch
from scipy.ndimage import gaussian_filter
from scipy import ndimage, stats
from astropy.coordinates import SkyCoord, EarthLocation, AltAz
from astropy.time import Time
import astropy.units as u
from lsst.resources import ResourcePath
from lsst.summit.utils import ConsDbClient

# ── rubin_scheduler ────────────────────────────────────────────────────────
try:
    from rubin_scheduler.site_models import SeeingModel
except ImportError:
    from rubin_scheduler.scheduler.utils import SeeingModel

try:
    from rubin_scheduler.utils import m5_flat_sed as _m5_raw
except ImportError:
    from rubin_scheduler.scheduler.utils import m5_flat_sed as _m5_raw

_M5_PARAMS = list(inspect.signature(_m5_raw).parameters.keys())
_HAS_TAU   = "tau_cloud" in _M5_PARAMS
print(f"[rubin_scheduler] m5_flat_sed params: {_M5_PARAMS}")
print(f"[rubin_scheduler] native tau_cloud support: {_HAS_TAU}")

warnings.filterwarnings("ignore")

os.environ.setdefault("no_proxy", "")
if ".consdb" not in os.environ["no_proxy"]:
    os.environ["no_proxy"] += ",.consdb"

# ─────────────────────────────────────────────────────────────────────────────
# CONSTANTS  (geometry / slew — unchanged)
# ─────────────────────────────────────────────────────────────────────────────
NSIDE_EXPECTED = 32
NEST           = True
UNSEEN         = hp.UNSEEN if hasattr(hp, "UNSEEN") else np.nan

RUBIN_LAT      = -30.244639
RUBIN_LON      = -70.749417
RUBIN_HEIGHT_M = 2663.0

BIN_SIZE_KM    = 1000
R_PROJECTION   = 10000.0

CBAR_VMIN = -0.2
CBAR_VMAX  = 2.0

# Slew
MAX_SLEW_SPEED_ALT = 3.5
MAX_SLEW_SPEED_AZ  = 7.0
SLEW_SETTLE_TIME   = 2.0

# Exposure
EXPOSURE_TIME = 30.0
READOUT_TIME  = 2.0
SLOT_TIME     = EXPOSURE_TIME + READOUT_TIME

# Pixel quality
MAX_SIGMA_MAG  = 0.3
MAX_FLAG_VALUE = 0
MIN_ALT_DEG    = 15.0

# Patchiness
PATCHINESS_SAMPLE_N   = 10
PATCHINESS_LOW_FRAC   = 0.10
PATCHINESS_HIGH_FRAC  = 0.90
PATCHINESS_VAR_THRESH = 0.05

# ConsDB
CONSDB_URL   = "http://consdb-pq.consdb:8080/consdb"
INSTRUMENT   = "lsstcam"
N_ZP_SAMPLES = 20
ZP_SEED      = 42
ALPHA_SEEING = 0.6
BETA_SEEING  = 0.2
FWHM_CONSTANT = 2 * np.sqrt(2 * np.log(2))
EFFECTIVE_WAVELENGTHS = {
    'u': 366.3, 'g': 481.0, 'r': 622.2,
    'i': 754.5, 'z': 869.1, 'y': 971.0
}
FILENAME_BAND_MAP = {
    'B': None, 'u': 'u', 'g': 'g',
    'r': 'r',  'i': 'i', 'z': 'z', 'y': 'y',
}

URL_COL  = "lsst.sal.DREAM.logevent_largeFileObjectAvailable.url"
TIME_COL = "time"

# Default band assumed when per-slot band is unknown
DEFAULT_BAND = "r"

# ─────────────────────────────────────────────────────────────────────────────
# rubin_scheduler helpers
# ─────────────────────────────────────────────────────────────────────────────

_seeing_model_instance: SeeingModel | None = None

def _seeing_model() -> SeeingModel:
    global _seeing_model_instance
    if _seeing_model_instance is None:
        _seeing_model_instance = SeeingModel()
    return _seeing_model_instance


def compute_fwhm_eff(fwhm500: float, airmass: float, band: str) -> float:
    """SeeingModel → fwhmEff for one visit."""
    sm  = _seeing_model()
    res = sm(fwhm_z=float(fwhm500), airmass=float(airmass))
    try:
        idx = sm.band_list.index(band)
    except ValueError:
        idx = sm.band_list.index(DEFAULT_BAND)
    return float(res["fwhmEff"][idx])


def m5_corrected(band: str,
                 musky: float,
                 fwhm_eff: float,
                 exptime: float,
                 airmass: float,
                 tau_cloud_mag: float,
                 nexp: int = 1) -> tuple[float, float]:
    """
    Return (m5_clear, m5_cloud) using the correct native path.

    If this rubin_scheduler build exposes tau_cloud, we pass the DREAM
    extinction directly — this is the most physically correct route because
    m5_flat_sed then propagates the extinction through its internal noise
    model (sky + source + readout) correctly.

    If tau_cloud is absent (older builds) we fall back to the additive
    subtraction  m5_cloud = m5_clear − tau_cloud_mag,  which is a good
    approximation for tau < ~1 mag.

    Both calls are always positional to avoid kwarg-name version skew.
    """
    tau = float(tau_cloud_mag) if not np.isnan(tau_cloud_mag) else 0.0

    if _HAS_TAU:
        m5_cl  = float(_m5_raw(band, musky, fwhm_eff, exptime, airmass, nexp, 0.0))
        m5_cld = float(_m5_raw(band, musky, fwhm_eff, exptime, airmass, nexp, tau))
    else:
        m5_cl  = float(_m5_raw(band, musky, fwhm_eff, exptime, airmass, nexp))
        m5_cld = m5_cl - tau

    return m5_cl, (np.nan if np.isnan(tau_cloud_mag) else m5_cld)


# ─────────────────────────────────────────────────────────────────────────────
# OpSim loader  (called once per night inside run_night)
# ─────────────────────────────────────────────────────────────────────────────

_OPSIM_WANT = [
    "observationId", "observationStartMJD", "night",
    "filter",                      # band name
    "airmass",
    "seeingFwhm500",               # DIMM FWHM at zenith, 500 nm [arcsec]
    "skyBrightness",               # musky [mag/arcsec²]
    "visitExposureTime",           # shutter-open time [s]
    "numExposures",
    "fieldRA", "fieldDec",
    "seeingFwhmEff",               # OpSim's own value — for cross-check
    "fiveSigmaDepth",              # OpSim's own M5   — for cross-check
]


def _opsim_table(conn) -> str:
    cur = conn.cursor()
    cur.execute("SELECT name FROM sqlite_master WHERE type='table'")
    tables = [r[0] for r in cur.fetchall()]
    return next(
        (t for t in ["observations", "SummaryAllProps", "Summary", "obs"]
         if t in tables),
        tables[0])


def load_opsim_night(db_path: str, night: int) -> pd.DataFrame:
    """Load one night of OpSim rows; returns empty DF if DB missing."""
    if not os.path.exists(db_path):
        print(f"  [OpSim] DB not found: {db_path} — seeing correction skipped")
        return pd.DataFrame()
    conn  = sqlite3.connect(db_path)
    table = _opsim_table(conn)
    cur   = conn.cursor()
    cur.execute(f"PRAGMA table_info({table})")
    avail = {r[1] for r in cur.fetchall()}
    cols  = [c for c in _OPSIM_WANT if c in avail]
    sel   = ", ".join(f'"{c}"' for c in cols)
    night_col = next((c for c in avail if "night" in c.lower()), "night")
    df = pd.read_sql_query(
        f'SELECT {sel} FROM "{table}" WHERE "{night_col}" = {night}', conn)
    conn.close()
    df = df.rename(columns={
        "filter":            "band",
        "visitExposureTime": "exptime",
        "numExposures":      "nexp",
    })
    if "nexp"   not in df.columns: df["nexp"]   = 1
    if "exptime" not in df.columns: df["exptime"] = EXPOSURE_TIME
    print(f"  [OpSim] night {night}: {len(df)} obs loaded from '{table}'")
    return df


# Pre-computed once when load_opsim_night is called — stored as module-level
# so _nearest_opsim can normalise without extra args.
_OPSIM_MJD_MIN: float = np.nan
_OPSIM_MJD_MAX: float = np.nan
_DREAM_MJD_MIN: float = np.nan
_DREAM_MJD_MAX: float = np.nan


def _set_epoch_bounds(opsim_df: pd.DataFrame, dream_times: list) -> None:
    """Call once per night after loading OpSim + DREAM frames."""
    global _OPSIM_MJD_MIN, _OPSIM_MJD_MAX, _DREAM_MJD_MIN, _DREAM_MJD_MAX
    if not opsim_df.empty:
        _OPSIM_MJD_MIN = float(opsim_df["observationStartMJD"].min())
        _OPSIM_MJD_MAX = float(opsim_df["observationStartMJD"].max())
    if dream_times:
        from astropy.time import Time as _Time
        mjds = [_Time(t).mjd for t in dream_times]
        _DREAM_MJD_MIN = float(min(mjds))
        _DREAM_MJD_MAX = float(max(mjds))


def _nearest_opsim(opsim_df: pd.DataFrame,
                   mjd: float,
                   band: str) -> dict | None:
    """
    Match a DREAM frame (real MJD) to an OpSim row by fractional
    position within the night.

    OpSim nights are a *simulation* epoch — their absolute MJDs do not
    correspond to the real DREAM observation timestamps.  We therefore
    normalise each side to [0, 1] over its own night window and match
    on that fraction, so "30% through the night" always pairs correctly.

    Falls back to ignoring band if no same-band rows exist.
    Returns None only if opsim_df is empty.
    """
    if opsim_df.empty:
        return None

    # Fractional position of this DREAM frame in its observed night
    d_range = _DREAM_MJD_MAX - _DREAM_MJD_MIN
    if d_range > 0:
        frac = (mjd - _DREAM_MJD_MIN) / d_range
    else:
        frac = 0.5

    # Corresponding fractional MJD in the OpSim night
    o_range = _OPSIM_MJD_MAX - _OPSIM_MJD_MIN
    target_mjd = _OPSIM_MJD_MIN + frac * o_range if o_range > 0 else _OPSIM_MJD_MIN

    sub = opsim_df[opsim_df["band"] == band] if "band" in opsim_df.columns else opsim_df
    if sub.empty:
        sub = opsim_df

    dt  = (sub["observationStartMJD"] - target_mjd).abs()
    idx = dt.idxmin()
    return sub.loc[idx].to_dict()


# ─────────────────────────────────────────────────────────────────────────────
# URL helpers
# ─────────────────────────────────────────────────────────────────────────────

def transform_url(url: str) -> str:
    url = str(url).strip()
    if url.startswith("https://s3.cp.lsst.org/"):
        return url.replace("https://s3.cp.lsst.org/", "s3://lfa@")
    return url


def band_from_url(url: str):
    m = re.search(r'_([A-Za-z])_cloud_zps', url)
    return m.group(1) if m else None


# ─────────────────────────────────────────────────────────────────────────────
# HDF5 loaders
# ─────────────────────────────────────────────────────────────────────────────

def _load_hdf5(url: str):
    rp = ResourcePath(url)
    with rp.open("rb") as fd:
        data = fd.read()
    with h5py.File(io.BytesIO(data), "r") as f:
        clouds       = np.array(f["clouds"],       dtype=float).ravel()
        sigma        = np.array(f["sigma"],        dtype=float).ravel()
        flags        = np.array(f["flags"],        dtype=int  ).ravel()
        mask_visible = np.array(f["mask_visible"], dtype=bool ).ravel()
        nobs         = np.array(f["nobs"],         dtype=int  ).ravel()
    bad = (
        ~mask_visible | (nobs == 0)
        | (flags > MAX_FLAG_VALUE)
        | (sigma > MAX_SIGMA_MAG)
        | ~np.isfinite(clouds)
    )
    clouds[bad] = np.nan
    sigma [bad] = np.nan
    return clouds, sigma


fetch_sys_map = _load_hdf5
fetch_zps_map = _load_hdf5


# ─────────────────────────────────────────────────────────────────────────────
# Coordinate utilities  (unchanged from original)
# ─────────────────────────────────────────────────────────────────────────────

def _make_location():
    return EarthLocation(lat=RUBIN_LAT*u.deg,
                         lon=RUBIN_LON*u.deg,
                         height=RUBIN_HEIGHT_M*u.m)


def _ensure_time(t):
    if not isinstance(t, Time):
        t = Time(t)
    return t.utc


def radec_to_altaz(ra_deg, dec_deg, obstime):
    t   = _ensure_time(obstime)
    sky = SkyCoord(ra=ra_deg*u.deg, dec=dec_deg*u.deg, frame="icrs")
    aa  = sky.transform_to(AltAz(obstime=t, location=_make_location()))
    return float(aa.alt.deg), float(aa.az.deg % 360.0)


def altaz_to_xy(alt_deg, az_deg):
    if alt_deg < MIN_ALT_DEG:
        return None, None
    alt_r = np.radians(alt_deg)
    az_r  = np.radians(az_deg)
    scale = R_PROJECTION / np.sin(alt_r)
    return (-np.cos(alt_r)*np.sin(az_r)*scale,
             np.cos(alt_r)*np.cos(az_r)*scale)


def xy_to_altaz(x_km, y_km):
    r       = np.sqrt(x_km**2 + y_km**2)
    alt_deg = float(np.degrees(np.arctan2(R_PROJECTION, r)))
    az_deg  = float(np.degrees(np.arctan2(-x_km, y_km)) % 360.0)
    return alt_deg, az_deg


def xy_to_zenith_angle(x_km, y_km):
    alt_deg, _ = xy_to_altaz(x_km, y_km)
    return 90.0 - alt_deg


def radec_to_xy(ra_deg, dec_deg, obstime):
    alt, az = radec_to_altaz(ra_deg, dec_deg, obstime)
    x, y    = altaz_to_xy(alt, az)
    return x, y, alt, az


def healpix_to_altaz_vals(mp, nside, obstime):
    npix = hp.nside2npix(nside)
    pix  = np.arange(npix)
    theta, phi = hp.pix2ang(nside, pix, nest=NEST)
    ra  = np.degrees(phi)
    dec = 90.0 - np.degrees(theta)
    t   = _ensure_time(obstime)
    sky = SkyCoord(ra=ra*u.deg, dec=dec*u.deg, frame="icrs")
    aa  = sky.transform_to(AltAz(obstime=t, location=_make_location()))
    vals = np.where(np.asarray(mp) == UNSEEN, np.nan, np.asarray(mp, dtype=float))
    return aa.az.deg % 360.0, aa.alt.deg, vals


def _healpix_to_grid(mp, obstime, bin_size=BIN_SIZE_KM):
    az, alt, vals = healpix_to_altaz_vals(mp, NSIDE_EXPECTED, obstime)
    alt_r = np.radians(alt); az_r = np.radians(az)
    above = alt > MIN_ALT_DEG
    scale = np.where(above, R_PROJECTION / np.sin(alt_r), np.nan)
    xf = -np.cos(alt_r)*np.sin(az_r)*scale
    yf =  np.cos(alt_r)*np.cos(az_r)*scale
    vf = np.where(above, vals, np.nan)
    c  = np.sqrt(xf**2 + yf**2) <= 15000.0
    ok = ~np.isnan(vf[c])
    xe = np.arange(-15000, 15001, bin_size)
    ye = np.arange(-15000, 15001, bin_size)
    Hs, _, _ = np.histogram2d(xf[c][ok], yf[c][ok], bins=[xe, ye], weights=vf[c][ok])
    Hc, _, _ = np.histogram2d(xf[c][ok], yf[c][ok], bins=[xe, ye])
    with np.errstate(divide="ignore", invalid="ignore"):
        H = Hs / Hc
    H[Hc == 0] = np.nan
    H = H.T
    xc = (xe[:-1] + xe[1:]) / 2; yc = (ye[:-1] + ye[1:]) / 2
    Xg, Yg = np.meshgrid(xc, yc)
    H[np.sqrt(Xg**2 + Yg**2) > 15000] = np.nan
    return H, Xg, Yg, xe, ye


def process_to_grid(clouds, sigma, obstime, bin_size=BIN_SIZE_KM):
    ext_grid, Xg, Yg, xe, ye = _healpix_to_grid(clouds, obstime, bin_size)
    sig_grid, *_              = _healpix_to_grid(sigma,  obstime, bin_size)
    return ext_grid, sig_grid, Xg, Yg, xe, ye


# ─────────────────────────────────────────────────────────────────────────────
# Patchiness
# ─────────────────────────────────────────────────────────────────────────────

def compute_patchiness(grids):
    idx = np.linspace(0, len(grids)-1, min(PATCHINESS_SAMPLE_N, len(grids)), dtype=int)
    fracs, vars_ = [], []
    for i in idx:
        g = grids[i]; valid = ~np.isnan(g)
        fracs.append(np.sum(valid) / g.size)
        vars_.append(float(np.nanvar(g)) if np.sum(valid) > 10 else 0.0)
    mf, mv = float(np.mean(fracs)), float(np.mean(vars_))
    if mf < PATCHINESS_LOW_FRAC:
        return False, mf, mv, f"Mostly clear (frac={mf:.2f})"
    if mf > PATCHINESS_HIGH_FRAC:
        return False, mf, mv, f"Mostly clouded (frac={mf:.2f})"
    if mv < PATCHINESS_VAR_THRESH:
        return False, mf, mv, f"Uniform cover (var={mv:.4f})"
    return True, mf, mv, f"Patchy (frac={mf:.2f}, var={mv:.4f})"


def quick_patchiness_check(df_night, n_probe=PATCHINESS_SAMPLE_N):
    if len(df_night) == 0:
        return False, 0.0, 0.0, "No frames", []
    idx = np.linspace(0, len(df_night)-1, min(n_probe, len(df_night)), dtype=int)
    grids = []
    for i in idx:
        row = df_night.iloc[i]
        try:
            clouds, sigma = fetch_sys_map(transform_url(row[URL_COL]))
            ext_g, *_ = process_to_grid(clouds, sigma, row[TIME_COL].to_pydatetime())
            grids.append(ext_g)
        except Exception as e:
            print(f"    probe load failed: {e}")
    if len(grids) < 2:
        return False, 0.0, 0.0, "Too few probe frames", []
    patchy, mf, mv, reason = compute_patchiness(grids)
    return patchy, mf, mv, reason, grids


# ─────────────────────────────────────────────────────────────────────────────
# Slew  (unchanged)
# ─────────────────────────────────────────────────────────────────────────────

def calculate_slew_time(alt1, az1, alt2, az2):
    da  = abs(alt2 - alt1)
    daz = abs(az2 - az1)
    if daz > 180: daz = 360 - daz
    return max(da / MAX_SLEW_SPEED_ALT, daz / MAX_SLEW_SPEED_AZ) + SLEW_SETTLE_TIME


# ─────────────────────────────────────────────────────────────────────────────
# Grid helpers
# ─────────────────────────────────────────────────────────────────────────────

def get_value_at_position(grid, x_km, y_km):
    xi = int(round((x_km / BIN_SIZE_KM) + 15))
    yi = int(round((y_km / BIN_SIZE_KM) + 15))
    if 0 <= xi < grid.shape[1] and 0 <= yi < grid.shape[0]:
        return grid[yi, xi]
    return np.nan


def find_absolute_minimum(grid):
    if not np.any(~np.isnan(grid)):
        return 0, 0, np.nan
    idx = np.nanargmin(grid)
    yi, xi = np.unravel_index(idx, grid.shape)
    return (xi-15)*BIN_SIZE_KM, (yi-15)*BIN_SIZE_KM, grid[yi, xi]


# ─────────────────────────────────────────────────────────────────────────────
# Cloud motion tracking  (unchanged)
# ─────────────────────────────────────────────────────────────────────────────

def compute_motion_with_correlation(grid1, grid2, sigma=5.0, search_range=10):
    g1 = gaussian_filter(np.nan_to_num(grid1), sigma)
    g2 = gaussian_filter(np.nan_to_num(grid2), sigma)
    m1 = ~np.isnan(grid1) & (grid1 != 0)
    m2 = ~np.isnan(grid2) & (grid2 != 0)
    best_corr, best_dx, best_dy = -np.inf, 0, 0
    for dy in range(-search_range, search_range+1):
        for dx in range(-search_range, search_range+1):
            sh = ndimage.shift(g2, (dy, dx), order=1, mode="constant", cval=0)
            sm = ndimage.shift(m2.astype(float), (dy, dx), order=0,
                               mode="constant", cval=0) > 0.5
            val = m1 & sm
            if np.sum(val) < 100: continue
            v1, v2 = g1[val], sh[val]
            if np.std(v1) > 0 and np.std(v2) > 0:
                corr = np.corrcoef(v1, v2)[0, 1]
                if corr > best_corr:
                    best_corr, best_dx, best_dy = corr, dx, dy
    return best_dx, best_dy, best_corr


def predict_future_position(cx, cy, dx_pix, dy_pix, current_grid,
                             fallback_threshold=0.5):
    nx = cx - dx_pix * BIN_SIZE_KM
    ny = cy - dy_pix * BIN_SIZE_KM
    r  = np.sqrt(nx**2 + ny**2)
    if r > 14000: nx, ny = nx*14000/r, ny*14000/r
    val = get_value_at_position(current_grid, nx, ny)
    if np.isnan(val) or val > fallback_threshold:
        nx, ny, _ = find_absolute_minimum(current_grid)
    return nx, ny


def compute_all_positions(all_grids):
    x0, y0, _ = find_absolute_minimum(all_grids[0])
    abs_pos, pred_pos, motion_v = [(x0,y0)], [(x0,y0)], [(0,0)]
    xp, yp = x0, y0; HL = 3
    for i in range(1, len(all_grids)):
        if i % 50 == 0:
            print(f"  Tracking frame {i}/{len(all_grids)}")
        xa, ya, _ = find_absolute_minimum(all_grids[i])
        abs_pos.append((xa, ya))
        if i >= HL:
            dxs, dys = [], []
            for j in range(1, HL+1):
                dx, dy, conf = compute_motion_with_correlation(
                    all_grids[i-j], all_grids[i-j+1])
                if conf > 0.5: dxs.append(dx); dys.append(dy)
            dx_avg = float(np.mean(dxs)) if dxs else 0.0
            dy_avg = float(np.mean(dys)) if dys else 0.0
        else:
            dx_avg, dy_avg, _ = compute_motion_with_correlation(
                all_grids[i-1], all_grids[i])
        motion_v.append((dx_avg, dy_avg))
        xp, yp = predict_future_position(xp, yp, dx_avg, dy_avg, all_grids[i])
        pred_pos.append((xp, yp))
    return abs_pos, pred_pos, motion_v


# ─────────────────────────────────────────────────────────────────────────────
# Data loading
# ─────────────────────────────────────────────────────────────────────────────

def load_all_sys_frames(csv_file):
    df = pd.read_csv(csv_file)
    df.columns = df.columns.str.replace('"', "").str.strip()
    df = df.dropna(subset=[URL_COL]).copy()
    df[TIME_COL] = pd.to_datetime(df[TIME_COL], errors="coerce", utc=True)
    df = df.dropna(subset=[TIME_COL]).copy()
    df = df[df[URL_COL].str.contains(".hdf5",     na=False)].copy()
    df = df[df[URL_COL].str.contains("cloud_sys", na=False)].copy()
    df = df.sort_values(TIME_COL).reset_index(drop=True)
    df["night_key"] = (df[TIME_COL] - pd.Timedelta(hours=12)).dt.date
    return df


def load_all_zps_frames(csv_file):
    df = pd.read_csv(csv_file)
    df.columns = df.columns.str.replace('"', "").str.strip()
    df = df.dropna(subset=[URL_COL]).copy()
    df[TIME_COL] = pd.to_datetime(df[TIME_COL], errors="coerce", utc=True)
    df = df.dropna(subset=[TIME_COL]).copy()
    df = df[df[URL_COL].str.contains(".hdf5",     na=False)].copy()
    df = df[df[URL_COL].str.contains("cloud_zps", na=False)].copy()
    df = df.sort_values(TIME_COL).reset_index(drop=True)
    df["night_key"] = (df[TIME_COL] - pd.Timedelta(hours=12)).dt.date
    df["dream_band"] = df[URL_COL].apply(band_from_url)
    return df


def get_night_df(all_df, night_key):
    return all_df[all_df["night_key"] == night_key].copy().reset_index(drop=True)


def load_scheduler_observations(db_file):
    print("\n" + "="*70)
    print("LOADING SCHEDULER OBSERVATIONS")
    print("="*70)
    conn   = sqlite3.connect(db_file)
    cursor = conn.cursor()
    cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
    tables = [t[0] for t in cursor.fetchall()]
    table_name = next(
        (n for n in ["observations","SummaryAllProps","Summary","obs"] if n in tables),
        tables[0])
    print(f"Using table: {table_name}")
    obs_df = pd.read_sql_query(f"SELECT * FROM {table_name}", conn)
    conn.close()

    night_col = next((c for c in obs_df.columns if "night" in c.lower()), None)
    if night_col is None:
        raise RuntimeError("Could not find night column in scheduler DB")
    obs_df["night"] = obs_df[night_col].astype(int)
    for req in ["observationStartMJD","fieldRA","fieldDec"]:
        for col in obs_df.columns:
            if req.lower() in col.lower() and col != req:
                obs_df[req] = obs_df[col]

    night_scores = {}
    for night in obs_df["night"].unique()[:500]:
        no  = obs_df[obs_df["night"] == night]
        if len(no) < 200: continue
        dur = (no["observationStartMJD"].max() - no["observationStartMJD"].min()) * 24
        if dur < 5: continue
        night_scores[night] = (len(no)*10 + dur, len(no), dur)

    if not night_scores:
        selected = obs_df.groupby("night").size().idxmax()
    else:
        selected = max(night_scores, key=lambda k: night_scores[k][0])
        _, n, dur = night_scores[selected]
        print(f"  Night {selected}: {n} obs over {dur:.1f} hours")

    night_obs = obs_df[obs_df["night"] == selected].copy()
    night_obs = night_obs.sort_values("observationStartMJD").reset_index(drop=True)

    # Grab filter column if present (used for per-slot band lookup)
    for fcol in ["filter", "band", "visitFilter"]:
        if fcol in night_obs.columns and "band" not in night_obs.columns:
            night_obs["band"] = night_obs[fcol]

    print(f"Selected night {selected}: {len(night_obs)} observations")
    return night_obs, selected


def match_scheduler_to_frames(scheduler_obs, frame_times):
    n       = len(frame_times)
    fracs   = np.linspace(0, 1, n)
    mjd     = scheduler_obs["observationStartMJD"].values
    r       = mjd.max() - mjd.min()
    s_fracs = (mjd - mjd.min()) / r if r > 0 else np.zeros(len(mjd))
    matched = [scheduler_obs.iloc[np.argmin(np.abs(s_fracs - f))] for f in fracs]
    print(f"  Matched {len(matched)} frames to scheduler observations")
    return matched


# ─────────────────────────────────────────────────────────────────────────────
# Metrics  —  now with real M5
# ─────────────────────────────────────────────────────────────────────────────

def _empty_strategy():
    return {
        # geometry / slew
        "frame_indices": [], "alt_az": [],
        "zenith_angles": [], "slew_times": [], "expose_times": [],
        # cloud
        "extinctions": [], "sigma": [],
        # seeing (from OpSim via SeeingModel)
        "seeing500":  [],   # seeingFwhm500 pulled from OpSim  [arcsec]
        "fwhmEff":    [],   # SeeingModel output               [arcsec]
        "sky_bright": [],   # skyBrightness (musky) from OpSim [mag/arcsec²]
        # depth
        "m5_clear":     [],  # m5_flat_sed, tau=0
        "m5_corrected": [],  # m5_flat_sed, tau=cloud_ext  (or clear - ext)
        # legacy photon proxy kept for backward compat but not used for depth
        "photons": [],
    }


def _record_obs(metrics, key, ext_mag, sigma_val, za, slew_t,
                expose_t, frame_i, alt, az, prev_alt, prev_az,
                opsim_row, band):
    """
    Record one observation slot.  Calls SeeingModel + m5_flat_sed with real
    OpSim atmospheric quantities.  Falls back gracefully when OpSim is absent.
    """
    s = metrics[key]
    s["frame_indices"].append(frame_i)
    s["alt_az"].append((alt, az))
    s["zenith_angles"].append(za)
    s["slew_times"].append(slew_t)
    s["expose_times"].append(expose_t)
    s["extinctions"].append(ext_mag)
    s["sigma"].append(sigma_val)

    prev_alt[key] = alt
    prev_az[key]  = az

    # Legacy photon proxy (kept for video annotation only)
    if not np.isnan(ext_mag):
        flux = 10 ** (-0.4 * ext_mag)
        rate = 100.0 * np.pi*(6.4/2)**2 * 0.8 * flux
        s["photons"].append(rate * expose_t)
    else:
        s["photons"].append(np.nan)

    # ── Real seeing + M5 from OpSim ────────────────────────────────────────
    if opsim_row is not None:
        fwhm500 = float(opsim_row.get("seeingFwhm500", np.nan))
        musky   = float(opsim_row.get("skyBrightness",  np.nan))
        am      = float(opsim_row.get("airmass",        np.nan))
        expt    = float(opsim_row.get("exptime",        expose_t if expose_t > 0 else EXPOSURE_TIME))
        nexp    = int(  opsim_row.get("nexp",           1))
    else:
        fwhm500 = musky = am = np.nan
        expt = expose_t if expose_t > 0 else EXPOSURE_TIME
        nexp = 1

    s["seeing500"].append(fwhm500)
    s["sky_bright"].append(musky)

    if any(np.isnan(v) for v in [fwhm500, musky, am]):
        s["fwhmEff"].append(np.nan)
        s["m5_clear"].append(np.nan)
        s["m5_corrected"].append(np.nan)
        return

    feff = compute_fwhm_eff(fwhm500, am, band)
    s["fwhmEff"].append(feff)

    # Use the full slot exptime but guard against zero (slewed-out slot)
    effective_exptime = max(expose_t, 1.0) if expose_t > 0 else expt
    m5_cl, m5_cld = m5_corrected(band, musky, feff,
                                  effective_exptime, am, ext_mag, nexp)
    s["m5_clear"].append(m5_cl)
    s["m5_corrected"].append(m5_cld)


def compute_metrics(all_grids, all_sigma_grids, all_metas,
                    abs_positions, pred_positions, matched_scheduler,
                    opsim_df: pd.DataFrame,
                    default_band: str = DEFAULT_BAND):
    print("\n" + "="*70)
    print("COMPUTING METRICS  (OpSim-consistent seeing + native tau_cloud M5)")
    print("="*70)
    print(f"  Slot = {EXPOSURE_TIME}s exp + {READOUT_TIME}s readout = {SLOT_TIME}s")
    print(f"  tau_cloud native path: {_HAS_TAU}")
    print(f"  OpSim rows available:  {len(opsim_df)}")
    if not opsim_df.empty:
        bands_in_opsim = opsim_df["band"].unique().tolist() if "band" in opsim_df.columns else ["(no band col)"]
        print(f"  OpSim bands present:   {bands_in_opsim}")
        print(f"  OpSim MJD range:       {opsim_df['observationStartMJD'].min():.2f} – {opsim_df['observationStartMJD'].max():.2f}")

    metrics   = {s: _empty_strategy() for s in ("absolute","motion","scheduler")}
    prev_alt  = {s: None for s in metrics}
    prev_az   = {s: None for s in metrics}
    n_sched_rejected = 0

    for i in range(len(all_grids)):
        grid  = all_grids[i]
        sgrid = all_sigma_grids[i]
        t_py  = all_metas[i]["time"]
        mjd   = Time(t_py).mjd

        # ── Absolute minimum ──────────────────────────────────────────────
        xa, ya = abs_positions[i]
        alt_a, az_a = xy_to_altaz(xa, ya)
        if alt_a >= MIN_ALT_DEG:
            ext = get_value_at_position(grid,  xa, ya)
            sig = get_value_at_position(sgrid, xa, ya)
            za  = xy_to_zenith_angle(xa, ya)
            if not np.isnan(ext):
                slew_t   = (calculate_slew_time(prev_alt["absolute"], prev_az["absolute"],
                                                alt_a, az_a)
                            if prev_alt["absolute"] is not None else 0.0)
                expose_t = max(0.0, SLOT_TIME - slew_t)
                orow     = _nearest_opsim(opsim_df, mjd, default_band)
                _record_obs(metrics, "absolute", ext, sig, za,
                            slew_t, expose_t, i, alt_a, az_a,
                            prev_alt, prev_az, orow, default_band)

        # ── Motion tracking ───────────────────────────────────────────────
        xp, yp = pred_positions[i]
        alt_p, az_p = xy_to_altaz(xp, yp)
        if alt_p >= MIN_ALT_DEG:
            ext = get_value_at_position(grid,  xp, yp)
            sig = get_value_at_position(sgrid, xp, yp)
            za  = xy_to_zenith_angle(xp, yp)
            if not np.isnan(ext):
                slew_t   = (calculate_slew_time(prev_alt["motion"], prev_az["motion"],
                                                alt_p, az_p)
                            if prev_alt["motion"] is not None else 0.0)
                expose_t = max(0.0, SLOT_TIME - slew_t)
                orow     = _nearest_opsim(opsim_df, mjd, default_band)
                _record_obs(metrics, "motion", ext, sig, za,
                            slew_t, expose_t, i, alt_p, az_p,
                            prev_alt, prev_az, orow, default_band)

        # ── Scheduler ─────────────────────────────────────────────────────
        obs       = matched_scheduler[i]
        band_slot = str(obs.get("band", default_band)) if "band" in obs.index else default_band
        xs, ys, alt_s, az_s = radec_to_xy(
            float(obs["fieldRA"]), float(obs["fieldDec"]), t_py)
        if xs is None:
            n_sched_rejected += 1
            continue
        ext = get_value_at_position(grid,  xs, ys)
        sig = get_value_at_position(sgrid, xs, ys)
        za  = xy_to_zenith_angle(xs, ys)
        if not np.isnan(ext):
            slew_t   = (calculate_slew_time(prev_alt["scheduler"], prev_az["scheduler"],
                                            alt_s, az_s)
                        if prev_alt["scheduler"] is not None else 0.0)
            expose_t = max(0.0, SLOT_TIME - slew_t)
            orow     = _nearest_opsim(opsim_df, mjd, band_slot)
            _record_obs(metrics, "scheduler", ext, sig, za,
                        slew_t, expose_t, i, alt_s, az_s,
                        prev_alt, prev_az, orow, band_slot)

    if n_sched_rejected:
        print(f"  ⚠ Scheduler: {n_sched_rejected} frames below alt={MIN_ALT_DEG}°")
    return metrics


# ─────────────────────────────────────────────────────────────────────────────
# Statistics  —  report M5 instead of proxy photon depth
# ─────────────────────────────────────────────────────────────────────────────

def print_statistics(metrics):
    print("\n" + "="*70)
    print("NIGHT PERFORMANCE SUMMARY  (OpSim-consistent M5 depths)")
    print("="*70)
    for strategy in ("absolute","motion","scheduler"):
        s = metrics[strategy]
        if not s["frame_indices"]:
            print(f"\n{strategy.upper()}: no valid data"); continue

        expose_times  = np.array(s["expose_times"])
        slew_times    = np.array(s["slew_times"])
        m5c           = np.array(s["m5_corrected"])
        m5cl          = np.array(s["m5_clear"])
        fwhm          = np.array(s["fwhmEff"])
        ext           = np.array(s["extinctions"])
        sky           = np.array(s["sky_bright"])
        sig           = np.array(s["sigma"])

        total_exp   = np.sum(expose_times)
        total_slew  = np.sum(slew_times)
        total_slot  = len(expose_times) * SLOT_TIME
        eff         = total_exp / total_slot * 100 if total_slot > 0 else 0
        pct_lost    = np.mean(expose_times == 0) * 100

        valid_m5 = m5c[~np.isnan(m5c)]
        valid_cl = m5cl[~np.isnan(m5cl)]

        print(f"\n{strategy.upper()} STRATEGY:")
        print(f"  Number of slots:             {len(expose_times)}")
        print(f"  On-sky exposure time:        {total_exp:.0f} s  ({total_exp/3600:.2f} hr)")
        print(f"  Total slew time:             {total_slew:.0f} s  ({total_slew/3600:.2f} hr)")
        print(f"  Shutter-open efficiency:     {eff:.1f}%")
        print(f"  Slots lost to slew:          {pct_lost:.1f}%")
        print(f"  Mean extinction (DREAM):     {np.nanmean(ext):.3f} mag")
        print(f"  Mean pixel σ (DREAM):        {np.nanmean(sig):.4f} mag")
        print(f"  Mean seeingFwhm500 (OpSim):  {np.nanmean(s['seeing500']):.3f}\"")
        print(f"  Mean fwhmEff (SeeingModel):  {np.nanmean(fwhm):.3f}\"")
        print(f"  Mean skyBrightness (OpSim):  {np.nanmean(sky):.3f} mag/arcsec²")
        print(f"  Median m5_clear:             {np.median(valid_cl):.3f} mag" if len(valid_cl) else "  Median m5_clear:             N/A")
        print(f"  Median m5_corrected:         {np.median(valid_m5):.3f} mag" if len(valid_m5) else "  Median m5_corrected:         N/A")
        if len(valid_cl) and len(valid_m5):
            delta = np.nanmean(m5cl - m5c)
            print(f"  Mean cloud penalty:          {delta:.3f} mag  (clear − corrected)")


# ─────────────────────────────────────────────────────────────────────────────
# Metrics plot  —  depth panels now show real M5 instead of proxy
# ─────────────────────────────────────────────────────────────────────────────

def create_metrics_plot(metrics, output_file="metrics.png"):
    print("\n" + "="*70)
    print("CREATING METRICS PLOT")
    print("="*70)

    strategies = ("absolute","motion","scheduler")
    colors     = ("green","blue","red")
    labels     = ("Absolute Min","Motion Tracking","Scheduler")

    fig = plt.figure(figsize=(18, 14))
    gs  = fig.add_gridspec(4, 3, hspace=0.40, wspace=0.32)

    # ── Row 0: Zenith | Slew | σ ──────────────────────────────────────────
    ax = fig.add_subplot(gs[0, 0])
    for s, c, l in zip(strategies, colors, labels):
        if metrics[s]["zenith_angles"]:
            ax.hist(metrics[s]["zenith_angles"], bins=20, alpha=0.6,
                    color=c, label=l, edgecolor="black", linewidth=0.5)
    ax.set_xlabel("Zenith Angle (°)"); ax.set_ylabel("Count")
    ax.set_title("Zenith Angle Distribution", weight="bold")
    ax.legend(fontsize=9); ax.grid(alpha=0.3)

    ax = fig.add_subplot(gs[0, 1])
    for s, c, l in zip(strategies, colors, labels):
        slew = metrics[s]["slew_times"][1:]
        if slew:
            ax.hist(slew, bins=30, alpha=0.6, color=c, label=l,
                    edgecolor="black", linewidth=0.5)
    ax.set_xlabel("Slew Time (s)"); ax.set_ylabel("Count")
    ax.set_title("Slew Time Distribution", weight="bold")
    ax.legend(fontsize=9); ax.grid(alpha=0.3)

    ax = fig.add_subplot(gs[0, 2])
    for s, c, l in zip(strategies, colors, labels):
        sigs = [x for x in metrics[s]["sigma"] if not np.isnan(x)]
        if sigs:
            ax.hist(sigs, bins=30, alpha=0.6, color=c, label=l,
                    edgecolor="black", linewidth=0.5)
    ax.axvline(MAX_SIGMA_MAG, color="k", ls="--", lw=1.5,
               label=f"Mask threshold ({MAX_SIGMA_MAG})")
    ax.set_xlabel("Pixel σ (mag)"); ax.set_ylabel("Count")
    ax.set_title("Extinction Uncertainty", weight="bold")
    ax.legend(fontsize=9); ax.grid(alpha=0.3)

    # ── Row 1: fwhmEff | skyBrightness | m5_clear vs m5_corrected ────────
    ax = fig.add_subplot(gs[1, 0])
    for s, c, l in zip(strategies, colors, labels):
        fwhm = [x for x in metrics[s]["fwhmEff"] if not np.isnan(x)]
        if fwhm:
            ax.hist(fwhm, bins=25, alpha=0.6, color=c, label=l,
                    edgecolor="black", linewidth=0.5)
    ax.set_xlabel('fwhmEff (arcsec)'); ax.set_ylabel("Count")
    ax.set_title("Delivered FWHM_eff\n(SeeingModel from OpSim seeingFwhm500)",
                 weight="bold")
    ax.legend(fontsize=9); ax.grid(alpha=0.3)

    ax = fig.add_subplot(gs[1, 1])
    for s, c, l in zip(strategies, colors, labels):
        sky = [x for x in metrics[s]["sky_bright"] if not np.isnan(x)]
        if sky:
            ax.hist(sky, bins=25, alpha=0.6, color=c, label=l,
                    edgecolor="black", linewidth=0.5)
    ax.set_xlabel("Sky brightness (mag/arcsec²)"); ax.set_ylabel("Count")
    ax.set_title("Sky Brightness (musky)\nfrom OpSim — same as simulation",
                 weight="bold")
    ax.legend(fontsize=9); ax.grid(alpha=0.3)

    ax = fig.add_subplot(gs[1, 2])
    for s, c, l in zip(strategies, colors, labels):
        fi  = metrics[s]["frame_indices"]
        m5c = metrics[s]["m5_corrected"]
        m5l = metrics[s]["m5_clear"]
        if fi:
            ax.plot(fi, m5c, color=c, alpha=0.8, lw=1.2, label=f"{l} (corrected)")
            ax.plot(fi, m5l, color=c, alpha=0.3, lw=0.8, ls="--")
    ax.set_xlabel("Frame"); ax.set_ylabel("5σ depth (mag)")
    ax.set_title("m5_corrected (solid) vs m5_clear (dashed)\nover the night",
                 weight="bold")
    ax.legend(fontsize=8); ax.grid(alpha=0.3); ax.invert_yaxis()

    # ── Row 2: M5 timeline (full width) ──────────────────────────────────
    ax = fig.add_subplot(gs[2, :])
    for s, c, l in zip(strategies, colors, labels):
        fi  = metrics[s]["frame_indices"]
        m5c = metrics[s]["m5_corrected"]
        if fi:
            ax.plot(fi, m5c, color=c, alpha=0.8, lw=1.5, label=l)
    ax.set_xlabel("Frame Number (Time →)")
    ax.set_ylabel("Cloud-corrected 5σ depth (mag)")
    ax.set_title("Corrected M5 Over the Night\n"
                 "(m5_flat_sed with SeeingModel fwhmEff + OpSim musky + DREAM tau_cloud)",
                 weight="bold")
    ax.legend(fontsize=10); ax.grid(alpha=0.3); ax.invert_yaxis()

    # ── Row 3: Total exposure | Median corrected M5 | Efficiency ──────────
    ax = fig.add_subplot(gs[3, 0])
    exps = [np.sum(metrics[s]["expose_times"]) for s in strategies]
    bars = ax.bar(labels, exps, color=colors, alpha=0.7,
                  edgecolor="black", linewidth=1.5)
    baseline = exps[0] if exps[0] > 0 else 1
    for i, (e, bar) in enumerate(zip(exps, bars)):
        ax.text(i, e, f"{((e-baseline)/baseline)*100:+.1f}%",
                ha="center", va="bottom", fontsize=10, weight="bold")
    ax.set_ylabel("Total on-sky exposure (s)")
    ax.set_title("On-sky Exposure Time", weight="bold")
    ax.grid(alpha=0.3, axis="y")

    ax = fig.add_subplot(gs[3, 1])
    depths = []
    for s in strategies:
        m5c = np.array(metrics[s]["m5_corrected"])
        valid = m5c[~np.isnan(m5c)]
        depths.append(float(np.median(valid)) if len(valid) else np.nan)
    bars = ax.bar(labels, depths, color=colors, alpha=0.7,
                  edgecolor="black", linewidth=1.5)
    for bar, val in zip(bars, depths):
        if not np.isnan(val):
            ax.text(bar.get_x()+bar.get_width()/2, val, f"{val:.3f}",
                    ha="center", va="bottom", fontsize=10, weight="bold")
    ax.set_ylabel("Median corrected 5σ depth (mag)")
    ax.set_title("Survey Depth\n(m5_flat_sed + DREAM cloud)", weight="bold")
    ax.grid(alpha=0.3, axis="y"); ax.invert_yaxis()

    ax = fig.add_subplot(gs[3, 2])
    effs = []
    for s in strategies:
        et    = np.array(metrics[s]["expose_times"])
        slots = len(et) * SLOT_TIME
        effs.append(np.sum(et)/slots*100 if slots > 0 else 0)
    bars = ax.bar(labels, effs, color=colors, alpha=0.7,
                  edgecolor="black", linewidth=1.5)
    for bar, val in zip(bars, effs):
        ax.text(bar.get_x()+bar.get_width()/2, val, f"{val:.1f}%",
                ha="center", va="bottom", fontsize=10, weight="bold")
    ax.set_ylabel("Shutter-open efficiency (%)")
    ax.set_title("Observation Efficiency", weight="bold")
    ax.grid(alpha=0.3, axis="y"); ax.set_ylim(0, 105)

    plt.suptitle(
        "Rubin Observatory Pointing Strategy Comparison\n"
        "(OpSim-consistent: SeeingModel fwhmEff  |  OpSim musky  |  DREAM tau_cloud → m5_flat_sed)",
        fontsize=12, weight="bold", y=0.999)
    plt.savefig(output_file, dpi=150, bbox_inches="tight")
    print(f"  Metrics plot saved: {output_file}")
    plt.close()


# ─────────────────────────────────────────────────────────────────────────────
# Video  (unchanged layout — annotates m5_corrected instead of proxy)
# ─────────────────────────────────────────────────────────────────────────────

def create_video(all_grids, all_sigma_grids, all_metas,
                 abs_positions, pred_positions, motion_vectors,
                 matched_scheduler, x_edges, y_edges,
                 metrics,
                 output_file="cloud_tracking.mp4", fps=10):
    print("\n" + "="*70)
    print("CREATING VIDEO")
    print("="*70)

    # Build per-frame m5 lookup tables for annotation
    def _build_lookup(strategy):
        lut = {}
        for j, fi in enumerate(metrics[strategy]["frame_indices"]):
            lut[fi] = {
                "m5c":  metrics[strategy]["m5_corrected"][j],
                "fwhm": metrics[strategy]["fwhmEff"][j],
            }
        return lut
    lut_abs   = _build_lookup("absolute")
    lut_mot   = _build_lookup("motion")
    lut_sched = _build_lookup("scheduler")

    sched_xy = []
    for i, obs in enumerate(matched_scheduler):
        x, y, alt, az = radec_to_xy(float(obs["fieldRA"]), float(obs["fieldDec"]),
                                     all_metas[i]["time"])
        sched_xy.append((x, y, alt, az))

    fig, ax_map = plt.subplots(1, 1, figsize=(12, 10))

    def update(fi):
        ax_map.clear()
        grid  = all_grids[fi]
        sgrid = all_sigma_grids[fi]
        ts    = max(0, fi-12)

        ax_map.pcolormesh(x_edges, y_edges, grid, cmap="viridis",
                          vmin=CBAR_VMIN, vmax=CBAR_VMAX,
                          shading="flat", alpha=0.85)
        th = np.linspace(0, 2*np.pi, 300)
        ax_map.plot(15000*np.cos(th), 15000*np.sin(th), "k-", lw=2, alpha=0.5)
        ax_map.plot(0, 0, "r+", ms=15, mew=3, label="Zenith", zorder=5)

        def _m5_str(lut, fi):
            d = lut.get(fi, {})
            m = d.get("m5c", np.nan)
            f = d.get("fwhm", np.nan)
            ms = f"{m:.3f}" if not np.isnan(m) else "N/A"
            fs = f"{f:.3f}\"" if not np.isnan(f) else "N/A"
            return ms, fs

        # Green — absolute
        xa, ya = abs_positions[fi]
        va = get_value_at_position(grid, xa, ya)
        gx = [abs_positions[j][0] for j in range(ts, fi)]
        gy = [abs_positions[j][1] for j in range(ts, fi)]
        if len(gx) > 1: ax_map.plot(gx, gy, "g--", alpha=0.35, lw=1.5)
        ms_a, fw_a = _m5_str(lut_abs, fi)
        ax_map.plot(xa, ya, "go", ms=20, mew=2.5, mec="white", zorder=10,
                    label=f"Abs Min  ext={va:.3f}  m5={ms_a}  fwhm={fw_a}")

        # Blue — motion
        xp, yp = pred_positions[fi]
        vp = get_value_at_position(grid, xp, yp)
        bx = [pred_positions[j][0] for j in range(ts, fi)]
        by = [pred_positions[j][1] for j in range(ts, fi)]
        if len(bx) > 1: ax_map.plot(bx, by, "b--", alpha=0.35, lw=1.5)
        ms_m, fw_m = _m5_str(lut_mot, fi)
        ax_map.plot(xp, yp, "bo", ms=20, mew=2.5, mec="white", zorder=10,
                    label=f"Motion   ext={vp:.3f}  m5={ms_m}  fwhm={fw_m}")

        # Motion arrow
        if fi > 0:
            dx_pix, dy_pix = motion_vectors[fi]
            dxk, dyk = dx_pix*BIN_SIZE_KM, dy_pix*BIN_SIZE_KM
            mag = np.sqrt(dxk**2+dyk**2)
            if mag > 300:
                ax_map.add_patch(FancyArrowPatch(
                    (xp, yp), (xp+dxk, yp+dyk),
                    arrowstyle="->", mutation_scale=28, lw=2.5,
                    color="cyan", alpha=0.9, zorder=15))
                ax_map.text(xp+dxk+400, yp+dyk+400, f"{mag:.0f} km",
                            fontsize=9, color="cyan", weight="bold",
                            bbox=dict(boxstyle="round", fc="black", alpha=0.6))

        # Red — scheduler
        xs, ys, alt_s, _ = sched_xy[fi]
        ms_s, fw_s = _m5_str(lut_sched, fi)
        if xs is not None:
            vs = get_value_at_position(grid, xs, ys)
            ax_map.plot(xs, ys, "ro", ms=20, mew=2.5, mec="white", zorder=10,
                        label=f"Scheduler ext={vs:.3f}  m5={ms_s}  fwhm={fw_s}")
        else:
            ax_map.plot([], [], "ro", ms=20, mew=2.5, mec="white", alpha=0.3,
                        label=f"Scheduler (below horizon, alt={alt_s:.1f}°)")

        ts_str = all_metas[fi]["time"].strftime("%Y-%m-%d %H:%M:%S UTC")
        ax_map.set_xlabel("X (km)"); ax_map.set_ylabel("Y (km)")
        ax_map.set_aspect("equal"); ax_map.grid(alpha=0.2)
        ax_map.set_xlim(-16000,16000); ax_map.set_ylim(-16000,16000)
        ax_map.set_title(f"Cloud Coverage — {ts_str}\nFrame {fi+1}/{len(all_grids)}",
                         fontsize=12, weight="bold")
        ax_map.legend(loc="upper right", fontsize=8, framealpha=0.9)
        return []

    anim = animation.FuncAnimation(fig, update, frames=len(all_grids),
                                   interval=1000/fps, blit=False, repeat=False)
    Writer = animation.writers["ffmpeg"]
    writer = Writer(fps=fps, bitrate=5000, codec="libx264",
                    extra_args=["-pix_fmt","yuv420p"])
    anim.save(output_file, writer=writer)
    print(f"  Video saved: {output_file}")
    plt.close()


# ─────────────────────────────────────────────────────────────────────────────
# ZP cross-validation  (unchanged from original)
# ─────────────────────────────────────────────────────────────────────────────

def _safe_float(x):
    try: return float(x)
    except: return np.nan


def zps_value_at_radec(clouds, sigma_map, ra_deg, dec_deg, obstime):
    t   = _ensure_time(obstime)
    sky = SkyCoord(ra=ra_deg*u.deg, dec=dec_deg*u.deg, frame="icrs")
    aa  = sky.transform_to(AltAz(obstime=t, location=_make_location()))
    if aa.alt.deg < MIN_ALT_DEG:
        return np.nan, np.nan
    npix = hp.nside2npix(NSIDE_EXPECTED)
    pix  = np.arange(npix)
    th, ph = hp.pix2ang(NSIDE_EXPECTED, pix, nest=NEST)
    all_sky = SkyCoord(ra=np.degrees(ph)*u.deg,
                       dec=(90-np.degrees(th))*u.deg, frame="icrs")
    all_aa  = all_sky.transform_to(AltAz(obstime=t, location=_make_location()))
    v_alt = np.radians(aa.alt.deg); v_az = np.radians(aa.az.deg % 360)
    ar    = np.radians(all_aa.alt.deg); azr = np.radians(all_aa.az.deg % 360)
    cos_sep = (np.sin(v_alt)*np.sin(ar)
               + np.cos(v_alt)*np.cos(ar)*np.cos(v_az - azr))
    valid   = ~np.isnan(clouds) & (all_aa.alt.deg >= MIN_ALT_DEG)
    if not np.any(valid): return np.nan, np.nan
    best = np.nanargmax(np.where(valid, cos_sep, -np.inf))
    return float(clouds[best]), float(sigma_map[best])


def build_consdb_df(start_night, instrument=INSTRUMENT, consdb_url=CONSDB_URL):
    print("\n" + "="*70 + "\nBUILDING ConsDB DATAFRAME\n" + "="*70)
    client = ConsDbClient(consdb_url)
    t0     = pd.to_datetime(start_night)
    nights = np.arange(int(t0.strftime("%Y%m%d")),
                       int(pd.Timestamp.now().strftime("%Y%m%d")))
    dfs = []; zp_col = None
    for night in nights:
        try:
            v1 = client.query(
                f"SELECT * FROM cdb_{instrument}.visit1 WHERE day_obs = {night}"
            ).to_pandas()
            ql = client.query(
                f"SELECT * FROM cdb_{instrument}.visit1_quicklook WHERE day_obs = {night}"
            ).to_pandas()
            if ql.empty or v1.empty: continue
            merged = pd.merge(ql, v1, on="visit_id", how="inner",
                              suffixes=("_ql","_v1"))
            merged = merged[merged["airmass"].notnull() & (merged["airmass"] != 0.0)].copy()
            if merged.empty: continue
            if zp_col is None:
                for name in ["zeroPoint","zero_point","zp_mean","zeropoint",
                             "flux_mag0","fluxMag0"]:
                    if name in merged.columns: zp_col = name; break
                if zp_col is None:
                    cands = [c for c in merged.columns
                             if any(k in c.lower() for k in
                                    ["zeropoint","zero_point","zp","fluxmag0",
                                     "flux_mag0","photometric","calib"])]
                    if cands: zp_col = cands[0]
                print(f"  ZP column: '{zp_col}'")
            actual_zp = None
            if zp_col:
                for sfx in ("","_ql","_v1"):
                    if zp_col+sfx in merged.columns: actual_zp = zp_col+sfx; break
            merged["zp_consdb"] = (pd.to_numeric(merged[actual_zp], errors="coerce")
                                   if actual_zp else np.nan)
            merged["lambda_eff"]    = merged["band"].map(EFFECTIVE_WAVELENGTHS)
            merged["fwhm_observed"] = FWHM_CONSTANT * merged["psf_sigma_median"]
            merged["fwhm_zenith"]   = merged["fwhm_observed"] / (merged["airmass"]**ALPHA_SEEING)
            merged["seeing"]        = (merged["fwhm_zenith"]
                                       * (merged["lambda_eff"]/500.0)**BETA_SEEING)
            merged["obs_start"]    = pd.to_datetime(merged["obs_start"],
                                                     format="ISO8601", utc=True, errors="coerce")
            merged["obs_end"]      = pd.to_datetime(merged["obs_end"],
                                                     format="ISO8601", utc=True, errors="coerce")
            merged["obs_midpoint"] = (merged["obs_start"]
                                      + (merged["obs_end"]-merged["obs_start"])/2)
            for want, cands in {"ra":["s_ra","ra_v1","ra_ql","ra"],
                                 "dec":["s_dec","dec_v1","dec_ql","dec"]}.items():
                if want not in merged.columns:
                    for c in cands:
                        if c in merged.columns: merged[want] = merged[c]; break
            dfs.append(merged)
            print(f"  night {night}: {len(merged)} visits")
        except Exception as e:
            print(f"  night {night}: skipped ({e})")
    if not dfs:
        print("No ConsDB data found."); return pd.DataFrame()
    df = pd.concat(dfs, ignore_index=True)
    df = df[df["ra"].notna() & df["dec"].notna()].reset_index(drop=True)
    df["zp_consdb"] = pd.to_numeric(df["zp_consdb"], errors="coerce")
    print(f"ConsDB total: {len(df)} visits")
    return df


def cross_validate_zp(dream_zps_df, consdb_df,
                       n_samples=N_ZP_SAMPLES, seed=ZP_SEED):
    print("\n" + "="*70 + "\nZP CROSS-VALIDATION\n" + "="*70)
    if consdb_df.empty:
        print("No ConsDB data."); return pd.DataFrame()
    d_min, d_max = dream_zps_df[TIME_COL].min(), dream_zps_df[TIME_COL].max()
    c_min, c_max = consdb_df["obs_midpoint"].min(), consdb_df["obs_midpoint"].max()
    ov_s, ov_e   = max(d_min, c_min), min(d_max, c_max)
    if ov_s >= ov_e:
        print("No temporal overlap."); return pd.DataFrame()
    dream_ov  = dream_zps_df[(dream_zps_df[TIME_COL] >= ov_s)
                              & (dream_zps_df[TIME_COL] <= ov_e)].reset_index(drop=True)
    consdb_ov = consdb_df[(consdb_df["obs_midpoint"] >= ov_s)
                           & (consdb_df["obs_midpoint"] <= ov_e)].reset_index(drop=True)
    if dream_ov.empty or consdb_ov.empty: return pd.DataFrame()
    rng    = np.random.default_rng(seed)
    idx    = rng.choice(len(dream_ov), size=min(n_samples, len(dream_ov)), replace=False)
    sample = dream_ov.iloc[sorted(idx)].reset_index(drop=True)
    cadence= max((dream_ov[TIME_COL].max()-dream_ov[TIME_COL].min()).total_seconds()
                 / max(len(dream_ov), 1), 300.0)
    window = pd.Timedelta(seconds=cadence/2)
    rows   = []
    for _, frame in sample.iterrows():
        t_frame = frame[TIME_COL]
        url     = transform_url(frame[URL_COL])
        dbnd    = frame.get("dream_band")
        try:
            clouds, sigma_map = fetch_zps_map(url)
        except Exception as e:
            print(f"   load failed: {e}"); continue
        nearby = consdb_ov[(consdb_ov["obs_midpoint"] >= t_frame-window)
                            & (consdb_ov["obs_midpoint"] <= t_frame+window)].copy()
        lsst_band = FILENAME_BAND_MAP.get(dbnd)
        if lsst_band: nearby = nearby[nearby["band"] == lsst_band]
        for _, visit in nearby.iterrows():
            ra, dec = _safe_float(visit.get("ra")), _safe_float(visit.get("dec"))
            if np.isnan(ra) or np.isnan(dec): continue
            zp_d, sig_d = zps_value_at_radec(clouds, sigma_map, ra, dec, t_frame)
            zp_c        = _safe_float(visit.get("zp_consdb", np.nan))
            delta = (zp_d-zp_c if not (np.isnan(zp_d) or np.isnan(zp_c)) else np.nan)
            rows.append(dict(frame_time=t_frame, visit_id=visit.get("visit_id"),
                             band=str(visit.get("band","?")), dream_band=dbnd,
                             ra=ra, dec=dec,
                             airmass=_safe_float(visit.get("airmass",np.nan)),
                             seeing=_safe_float(visit.get("seeing",np.nan)),
                             zp_offset_dream=zp_d, sig_dream=sig_d,
                             zp_consdb=zp_c, delta=delta))
    return pd.DataFrame(rows)


def plot_zp_comparison(df, output="dream_zps_vs_consdb.png"):
    print("\n" + "="*70 + "\nZP COMPARISON PLOT\n" + "="*70)
    if df.empty: print("No data."); return
    for col in ["zp_offset_dream","sig_dream","zp_consdb","delta","airmass","seeing"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    valid = df.dropna(subset=["zp_offset_dream"])
    if valid.empty: print("No valid DREAM ZP offsets."); return
    bands  = sorted(valid["band"].dropna().unique())
    cmap   = plt.cm.get_cmap("tab10", max(len(bands),1))
    bc     = {b: cmap(i) for i, b in enumerate(bands)}
    has_cdb = valid["zp_consdb"].notna().any()
    ncols   = 3 if has_cdb else 2
    fig, axes = plt.subplots(1, ncols, figsize=(7*ncols, 5))
    fig.suptitle("DREAM .zps ZP Offset vs ConsDB", fontsize=13, weight="bold")
    ax = axes[0]
    for b in bands:
        s = valid[valid["band"]==b].dropna(subset=["airmass"])
        ax.errorbar(s["airmass"], s["zp_offset_dream"], yerr=s["sig_dream"],
                    fmt="o", ms=6, color=bc[b], label=b, alpha=0.8, capsize=3)
    ax.axhline(0,color="k",ls="--",lw=1.5); ax.set_xlabel("Airmass")
    ax.set_ylabel("DREAM ZP offset (mag)"); ax.set_title("ZP offset vs Airmass")
    ax.legend(fontsize=8); ax.grid(alpha=0.3)
    ax = axes[1]
    for b in bands:
        s = valid[valid["band"]==b]
        ax.scatter(s["frame_time"], s["zp_offset_dream"],
                   color=bc[b], s=40, alpha=0.9, label=b)
    ax.axhline(0,color="k",ls="--",lw=1.5); ax.set_ylabel("DREAM ZP offset (mag)")
    ax.set_title("ZP offset over night")
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
    ax.legend(fontsize=8); ax.grid(alpha=0.3)
    if has_cdb:
        valid2 = valid.dropna(subset=["zp_consdb"])
        ax = axes[2]
        for b in bands:
            s = valid2[valid2["band"]==b]
            if s.empty: continue
            ax.errorbar(s["zp_consdb"], s["zp_offset_dream"], yerr=s["sig_dream"],
                        fmt="o", ms=6, color=bc[b], label=b, alpha=0.8, capsize=3)
        if len(valid2) >= 2:
            sl,ic,r,*_ = stats.linregress(valid2["zp_consdb"].astype(float),
                                          valid2["zp_offset_dream"].astype(float))
            xs = np.linspace(valid2["zp_consdb"].min(), valid2["zp_consdb"].max(), 100)
            ax.plot(xs, sl*xs+ic, "r-", lw=1.5, label=f"slope={sl:.2f}  r={r:.2f}")
        ax.set_xlabel("ConsDB ZP"); ax.set_ylabel("DREAM ZP offset (mag)")
        ax.set_title("DREAM offset vs ConsDB ZP")
        ax.legend(fontsize=8); ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(output, dpi=150, bbox_inches="tight")
    print(f"  ZP plot saved: {output}")
    plt.close()


# ─────────────────────────────────────────────────────────────────────────────
# run_night  —  now passes OpSim DF into compute_metrics
# ─────────────────────────────────────────────────────────────────────────────

def run_night(night_key, df_sys_night, df_zps_night,
              scheduler_obs, opsim_night: int,
              consdb_df, opsim_db: str,
              output_dir=".", fps=10, max_frames=None,
              default_band=DEFAULT_BAND):

    night_str  = str(night_key)
    tag        = night_str.replace("-","")
    plot_file  = os.path.join(output_dir, f"metrics_{tag}.png")
    video_file = os.path.join(output_dir, f"cloud_tracking_{tag}.mp4")
    zp_file    = os.path.join(output_dir, f"dream_zps_vs_consdb_{tag}.png")

    print("\n" + "█"*70)
    print(f"  NIGHT: {night_str}  ({len(df_sys_night)} sys frames)")
    print("█"*70)

    if len(df_sys_night) < 5:
        print("  Too few frames — skipping."); return None

    # ── Patchiness probe ──────────────────────────────────────────────────
    print(f"\n  [Patchiness probe — {PATCHINESS_SAMPLE_N} frames]")
    patchy, mf, mv, reason, _ = quick_patchiness_check(df_sys_night)
    print(f"    valid_frac={mf:.3f}  var={mv:.4f} mag²  → {reason}")
    if not patchy:
        print("  Skipping (not patchy)."); return None

    # ── Load OpSim for this night ─────────────────────────────────────────
    opsim_df = load_opsim_night(opsim_db, opsim_night)

    # ── Load all DREAM frames ─────────────────────────────────────────────
    all_grids, all_sigma, all_metas = [], [], []
    x_edges = y_edges = None
    rows = (df_sys_night.iloc[:max_frames] if max_frames else df_sys_night).iterrows()
    for _, row in rows:
        url = transform_url(row[URL_COL])
        try:
            clouds, sigma = fetch_sys_map(url)
            meta = {"url": url, "time": row[TIME_COL].to_pydatetime()}
            ext_g, sig_g, _, _, xe, ye = process_to_grid(clouds, sigma, meta["time"])
            all_grids.append(ext_g); all_sigma.append(sig_g); all_metas.append(meta)
            if x_edges is None: x_edges, y_edges = xe, ye
            if len(all_grids) % 100 == 0:
                print(f"    Loaded {len(all_grids)} frames …")
        except Exception as e:
            print(f"    Frame load failed: {e}")
    print(f"  Loaded {len(all_grids)} frames total")
    if len(all_grids) < 2:
        print("  Too few frames — skipping."); return None

    # Register epoch bounds so _nearest_opsim can normalise correctly
    _set_epoch_bounds(opsim_df, [m["time"] for m in all_metas])
    if not np.isnan(_OPSIM_MJD_MIN):
        print(f"  [OpSim] MJD range: {_OPSIM_MJD_MIN:.2f} – {_OPSIM_MJD_MAX:.2f}")
        print(f"  [DREAM] MJD range: {_DREAM_MJD_MIN:.2f} – {_DREAM_MJD_MAX:.2f}")

    # Full-set patchiness re-check
    patchy2, mf2, mv2, reason2 = compute_patchiness(all_grids)
    print(f"  Full-set patchiness: frac={mf2:.3f}  var={mv2:.4f}  → {reason2}")
    if not patchy2:
        print("  Full-set check failed — skipping."); return None

    # ── Scheduler matching ────────────────────────────────────────────────
    matched_sched = match_scheduler_to_frames(
        scheduler_obs, [m["time"] for m in all_metas])

    # ── Cloud motion tracking ─────────────────────────────────────────────
    print(f"\n  [Cloud motion tracking — {len(all_grids)} frames]")
    abs_pos, pred_pos, motion_v = compute_all_positions(all_grids)

    # ── Metrics (with OpSim-consistent seeing + M5) ───────────────────────
    metrics = compute_metrics(
        all_grids, all_sigma, all_metas,
        abs_pos, pred_pos, matched_sched,
        opsim_df=opsim_df,
        default_band=default_band)
    print_statistics(metrics)

    # ── Plots + video ─────────────────────────────────────────────────────
    create_metrics_plot(metrics, output_file=plot_file)
    create_video(all_grids, all_sigma, all_metas,
                 abs_pos, pred_pos, motion_v,
                 matched_sched, x_edges, y_edges,
                 metrics,
                 output_file=video_file, fps=fps)

    # ── ZP cross-validation ───────────────────────────────────────────────
    if not df_zps_night.empty and not consdb_df.empty:
        print(f"\n  [ZP cross-validation — {len(df_zps_night)} zps frames]")
        comparison = cross_validate_zp(df_zps_night, consdb_df)
        if not comparison.empty:
            for col in ["zp_offset_dream","sig_dream","zp_consdb","delta","airmass"]:
                comparison[col] = pd.to_numeric(comparison[col], errors="coerce")
            valid_zp = comparison.dropna(subset=["zp_offset_dream"])
            print(f"  Mean DREAM ZP offset: {valid_zp['zp_offset_dream'].mean():.4f} mag")
            print(f"  Std  DREAM ZP offset: {valid_zp['zp_offset_dream'].std():.4f} mag")
            valid2 = comparison.dropna(subset=["zp_offset_dream","zp_consdb"])
            if len(valid2) >= 2:
                r, p = stats.pearsonr(valid2["zp_offset_dream"].astype(float),
                                      valid2["zp_consdb"].astype(float))
                print(f"  Pearson r: {r:.3f}  (p={p:.4f})")
            plot_zp_comparison(comparison, output=zp_file)
        else:
            print("  No matched ZP pairs.")
    else:
        print("  Skipping ZP validation.")

    print(f"\n  ✓ Night {night_str} outputs: {plot_file}  |  {video_file}  |  {zp_file}")
    return metrics


# ─────────────────────────────────────────────────────────────────────────────
# main
# ─────────────────────────────────────────────────────────────────────────────

def main(
    csv_file     = "feb5_data.csv",
    opsim_db     = "baseline_v5.1.0_10yrs.db",   # OpSim v5.1 SQLite
    consdb_start = "2025-05-01",
    output_dir   = ".",
    fps          = 10,
    max_frames   = None,
    default_band = DEFAULT_BAND,
):
    """
    Iterate over every distinct night in the DREAM CSV.
    OpSim rows for the matched scheduler night are loaded once per DREAM night
    and forwarded to compute_metrics for seeing + M5 correction.
    """
    print("\n" + "="*70)
    print("DREAM CLOUD ANALYSIS PIPELINE  —  OpSim-consistent M5")
    print("="*70)
    print(f"  CSV      : {csv_file}")
    print(f"  OpSim DB : {opsim_db}")
    os.makedirs(output_dir, exist_ok=True)

    print("\nIndexing CSV …")
    all_sys = load_all_sys_frames(csv_file)
    all_zps = load_all_zps_frames(csv_file)
    all_nights = sorted(all_sys["night_key"].unique())
    print(f"Found {len(all_nights)} distinct nights")

    # Scheduler (baseline) — loaded once; provides the opsim_night index
    scheduler_obs, opsim_night = load_scheduler_observations(opsim_db)

    print("\nLoading ConsDB …")
    consdb_df = build_consdb_df(consdb_start)

    summary = []
    for night_key in all_nights:
        df_sys_night = get_night_df(all_sys, night_key)
        df_zps_night = get_night_df(all_zps, night_key)

        metrics = run_night(
            night_key, df_sys_night, df_zps_night,
            scheduler_obs, opsim_night,
            consdb_df, opsim_db,
            output_dir=output_dir, fps=fps,
            max_frames=max_frames, default_band=default_band)

        if metrics is not None:
            row = {"night": str(night_key)}
            for s in ("absolute","motion","scheduler"):
                m5c = np.array(metrics[s]["m5_corrected"])
                valid = m5c[~np.isnan(m5c)]
                row[f"median_m5_{s}"]  = float(np.median(valid)) if len(valid) else np.nan
                row[f"n_slots_{s}"]    = len(metrics[s]["frame_indices"])
                row[f"mean_fwhm_{s}"]  = float(np.nanmean(metrics[s]["fwhmEff"]))
                row[f"mean_ext_{s}"]   = float(np.nanmean(metrics[s]["extinctions"]))
            summary.append(row)

    print("\n" + "="*70 + "\nALL-NIGHTS SUMMARY\n" + "="*70)
    if summary:
        df_sum = pd.DataFrame(summary)
        print(df_sum.to_string(index=False))
        sum_file = os.path.join(output_dir, "all_nights_summary.csv")
        df_sum.to_csv(sum_file, index=False)
        print(f"\nSummary → {sum_file}")
        n_total = len(all_nights)
        n_patchy = len(summary)
        print(f"Nights checked: {n_total}  |  patchy (run): {n_patchy}  "
              f"|  skipped: {n_total-n_patchy}")
    else:
        print("No patchy nights found.")
    print("\nDone.")
    return summary


if __name__ == "__main__":
    main()

[rubin_scheduler] m5_flat_sed params: ['visit_filter', 'musky', 'fwhm_eff', 'exp_time', 'airmass', 'nexp', 'tau_cloud']
[rubin_scheduler] native tau_cloud support: True

DREAM CLOUD ANALYSIS PIPELINE  —  OpSim-consistent M5
  CSV      : feb5_data.csv
  OpSim DB : baseline_v5.1.0_10yrs.db

Indexing CSV …
Found 189 distinct nights

LOADING SCHEDULER OBSERVATIONS
Using table: observations
  Night 572: 1139 obs over 11.7 hours
Selected night 572: 1139 observations

Loading ConsDB …

BUILDING ConsDB DATAFRAME
  ZP column: 'eff_time_zero_point_scale_min'
  night 20250501: 591 visits
  night 20250502: 648 visits
  night 20250503: 619 visits
  night 20250504: 512 visits
  night 20250505: 459 visits
  night 20250506: 295 visits
  night 20250507: 227 visits
  night 20250510: 71 visits
  night 20250511: 585 visits
  night 20250512: 608 visits
  night 20250513: 502 visits
  night 20250515: 417 visits
  night 20250519: 504 visits
  night 20250520: 302 visits
  night 20250521: 293 visits
  night 202